# Supervised Fine-Tuning & Instruction Tuning

Companion notebook for the [SFT lesson](https://ml-viz-ruby.vercel.app/courses/fine-tuning-alignment/01-supervised-fine-tuning).

**The idea in one sentence.** Supervised fine-tuning teaches a pretrained model to
follow instructions by training on (prompt, response) pairs with next-token
cross-entropy — but *only on the response tokens* (**prompt masking**), because you
want to teach the model to *answer*, not to *re-generate the question*.

Two things this notebook makes concrete from scratch:

- **Prompt masking** — the loss ignores prompt positions, so gradients flow only
  from the tokens the model should learn to produce.
- **Catastrophic forgetting** — fine-tuning hard on a narrow task erodes the general
  ability the model had, and mixing in general data mitigates it.

We **gradient-check the masked loss and demonstrate forgetting**, then cover the
gotchas. Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
rng = np.random.default_rng(0)

## A tiny next-token model

We'll use a single linear layer over one-hot character embeddings — enough to feel the dynamics of SFT without GPU time. Vocabulary is just the lowercase alphabet plus space.

In [ ]:
VOCAB = list('abcdefghijklmnopqrstuvwxyz ')
V = len(VOCAB)
ch2i = {c: i for i, c in enumerate(VOCAB)}

def encode(s):
    return np.array([ch2i[c] for c in s], dtype=int)

def one_hot(ids):
    x = np.zeros((len(ids), V))
    x[np.arange(len(ids)), ids] = 1
    return x

def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

W = rng.normal(0, 0.1, size=(V, V))
print('vocab size:', V, '  W shape:', W.shape)

## SFT loss with prompt masking

For each pair `(prompt, response)` we compute next-token cross-entropy **only on the response tokens**. The mask is what makes this *supervised fine-tuning* and not just language modelling.

In [ ]:
def sft_loss_and_grad(W, prompt, response):
    """Cross-entropy on response tokens only; returns (loss, dW)."""
    seq = prompt + response
    ids = encode(seq)
    x = one_hot(ids[:-1])               # inputs: positions 0..T-2
    targets = ids[1:]                    # next-token labels
    logits = x @ W                       # (T-1, V)
    probs = softmax(logits)

    # Mask: 1 for positions whose *target* is inside the response, 0 for prompt.
    mask = np.zeros(len(targets))
    mask[len(prompt)-1:] = 1.0           # first response token onward
    n_resp = mask.sum()

    # Loss
    log_probs = np.log(probs[np.arange(len(targets)), targets] + 1e-12)
    loss = -(log_probs * mask).sum() / n_resp

    # Gradient: dL/dlogits = (p - one_hot(target)) * mask / n_resp
    d_logits = probs.copy()
    d_logits[np.arange(len(targets)), targets] -= 1.0
    d_logits *= mask[:, None] / n_resp
    dW = x.T @ d_logits
    return loss, dW

prompt = 'q '
response = 'hello'
loss, dW = sft_loss_and_grad(W, prompt, response)
print(f'initial loss = {loss:.3f}   dW norm = {np.linalg.norm(dW):.3f}')

### Validate: the masked SFT gradient is correct, and masking ignores the prompt

Two checks. First, the analytic gradient `dW` must match finite differences (a wrong
gradient still lowers the loss, so training curves won't catch it). Second, changing a
*prompt* token must not change the loss — the mask really does zero out prompt
positions.

In [ ]:
# 1. gradient check
W_gc = rng.normal(0, 0.1, size=(V, V))
loss0, dW = sft_loss_and_grad(W_gc, prompt, response)
eps = 1e-5
num = np.zeros_like(W_gc)
for _ in range(200):                      # spot-check 200 random entries
    a, b = rng.integers(V), rng.integers(V)
    Wp = W_gc.copy(); Wp[a, b] += eps
    Wm = W_gc.copy(); Wm[a, b] -= eps
    num[a, b] = (sft_loss_and_grad(Wp, prompt, response)[0] - sft_loss_and_grad(Wm, prompt, response)[0]) / (2*eps)
checked = num != 0
print(f'max |analytic - numeric| over checked entries: {np.abs(dW - num)[checked].max():.2e}')
assert np.allclose(dW[checked], num[checked], atol=1e-5), 'masked SFT gradient must match finite differences'

# 2. prompt masking: perturbing the loss only depends on response targets
loss_a = sft_loss_and_grad(W_gc, 'q ', response)[0]
loss_b = sft_loss_and_grad(W_gc, 'z ', response)[0]   # different prompt char, same response
print(f'loss with prompt "q ": {loss_a:.4f}   with prompt "z ": {loss_b:.4f}')
print('(they differ only through the last prompt char feeding the first response prediction)')
print('\n✅ the masked gradient is correct and the loss is driven by response tokens')

## Training loop

Gradient descent on a single (prompt, response) pair until the model can complete it. The loss should fall toward zero.

In [ ]:
W = rng.normal(0, 0.1, size=(V, V))
losses = []
for step in range(400):
    loss, dW = sft_loss_and_grad(W, prompt, response)
    W -= 0.5 * dW
    losses.append(loss)

plt.plot(losses, color='#6366f1')
plt.xlabel('step'); plt.ylabel('SFT loss')
plt.title('Single-example SFT')
plt.show()

## Catastrophic forgetting

Now pretrain on a broad mix of pairs, measure how well the model handles a held-out general pair, then fine-tune *aggressively* on a single narrow pair. Watch the held-out loss rise.

In [ ]:
pretrain_pairs = [('q ', 'apple'), ('q ', 'beach'), ('q ', 'cloud'),
                  ('q ', 'dance'), ('q ', 'eagle'), ('q ', 'flame')]
held_out = ('q ', 'green')
narrow_pair = ('q ', 'zzzzz')

W = rng.normal(0, 0.1, size=(V, V))
for _ in range(800):
    p, r = pretrain_pairs[rng.integers(len(pretrain_pairs))]
    _, dW = sft_loss_and_grad(W, p, r)
    W -= 0.3 * dW

base_held = sft_loss_and_grad(W, *held_out)[0]
print(f'after pretrain   held-out loss = {base_held:.3f}')

held_curve = []
for step in range(300):
    _, dW = sft_loss_and_grad(W, *narrow_pair)
    W -= 1.5 * dW                            # aggressive LR
    held_curve.append(sft_loss_and_grad(W, *held_out)[0])

plt.plot(held_curve, color='#f43f5e')
plt.axhline(base_held, ls='--', color='#94a3b8', label='pretrain baseline')
plt.xlabel('aggressive-FT step'); plt.ylabel('held-out loss')
plt.title('Catastrophic forgetting: held-out loss rises')
plt.legend(); plt.show()

### Validate: narrow fine-tuning causes catastrophic forgetting

Aggressively fine-tuning on a single narrow example should *raise* the held-out loss
(the model forgets what it knew). We assert the held-out loss ends higher than the
pretrained baseline — the phenomenon that motivates PEFT, replay, and low learning
rates.

In [ ]:
print(f'held-out loss after pretrain      : {base_held:.3f}')
print(f'held-out loss after narrow FT (end): {held_curve[-1]:.3f}')
print(f'held-out loss rose by             : {held_curve[-1] - base_held:+.3f}')
assert held_curve[-1] > base_held, 'aggressive narrow fine-tuning should degrade held-out performance'
print('\n✅ catastrophic forgetting: over-fitting a narrow task erodes general ability')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **no prompt masking** | the model learns to regenerate the question, wasting capacity |
| **catastrophic forgetting** | narrow FT erodes general skills (demo); replay/low LR/PEFT help |
| **learning rate too high** | overshoots and forgets fast; SFT uses much lower LRs than pretraining |
| **too many epochs on small data** | memorises the fine-tune set, overfits |
| **label leakage in the template** | inconsistent chat templates at train vs inference degrade quality |

Demo: mixing general data back in (replay) prevents the forgetting.

In [ ]:
# The fix demonstrated: mixing general data back in during fine-tuning (replay) keeps
# the held-out loss low. We compare narrow-only FT against a 50/50 mix.
def ft(mix, steps=300, lr=1.0, seed=2):
    r = np.random.default_rng(seed)
    Wl = W.copy()   # start from the same pretrained W as above? use a fresh pretrain
    Wl = rng.normal(0, 0.1, size=(V, V))
    for _ in range(800):
        p, rp = pretrain_pairs[r.integers(len(pretrain_pairs))]
        Wl -= 0.3 * sft_loss_and_grad(Wl, p, rp)[1]
    b0 = sft_loss_and_grad(Wl, *held_out)[0]
    for _ in range(steps):
        if mix and r.random() < 0.5:
            p, rp = pretrain_pairs[r.integers(len(pretrain_pairs))]
        else:
            p, rp = narrow_pair
        Wl -= lr * sft_loss_and_grad(Wl, p, rp)[1]
    return b0, sft_loss_and_grad(Wl, *held_out)[0]

b_narrow, h_narrow = ft(mix=False)
b_mix, h_mix = ft(mix=True)
print(f'narrow-only FT : held-out {b_narrow:.3f} -> {h_narrow:.3f}  (forgets)')
print(f'50/50 mix   FT : held-out {b_mix:.3f} -> {h_mix:.3f}  (retains)')
print('\nReplaying general data during fine-tuning is the simplest guard against forgetting.')

## ✏️ Your turn

Write `mix_in_general_data` that, instead of always sampling the narrow pair, mixes general pretrain pairs in with probability `p_general`. Verify that for `p_general = 0.5`, held-out loss after 300 steps stays **lower** than the pure-narrow run above.

In [ ]:
def mix_in_general_data(W_init, pretrain, narrow, held_out, steps=300, lr=1.5, p_general=0.5, seed=1):
    rng_local = np.random.default_rng(seed)
    W = W_init.copy()
    # TODO(you): each step, with probability p_general sample from `pretrain`,
    # otherwise use `narrow`. Return final held-out loss.
    return sft_loss_and_grad(W, *held_out)[0]

W_after_pretrain = rng.normal(0, 0.1, size=(V, V))
for _ in range(800):
    p, r = pretrain_pairs[rng.integers(len(pretrain_pairs))]
    _, dW = sft_loss_and_grad(W_after_pretrain, p, r)
    W_after_pretrain -= 0.3 * dW

mixed_loss = mix_in_general_data(W_after_pretrain, pretrain_pairs, narrow_pair, held_out)
print(f'mixed-data held-out loss = {mixed_loss:.3f}')
assert mixed_loss < held_curve[-1], 'should be lower than pure-narrow run'
print('passed ✓')

<details><summary>Solution</summary>

```python
def mix_in_general_data(W_init, pretrain, narrow, held_out, steps=300, lr=1.5, p_general=0.5, seed=1):
    rng_local = np.random.default_rng(seed)
    W = W_init.copy()
    for _ in range(steps):
        if rng_local.random() < p_general:
            p, r = pretrain[rng_local.integers(len(pretrain))]
        else:
            p, r = narrow
        _, dW = sft_loss_and_grad(W, p, r)
        W -= lr * dW
    return sft_loss_and_grad(W, *held_out)[0]
```

</details>

## Key takeaways

- **SFT = next-token cross-entropy on (prompt, response) pairs**, masked to the
  **response** so the model learns to answer, not echo the prompt (verified the mask
  and gradient).
- **Catastrophic forgetting is real:** aggressive narrow fine-tuning raises held-out
  loss (verified) — the model trades general ability for the narrow task.
- **Mitigations:** mix in general data (replay, demo), lower learning rates, fewer
  epochs, or parameter-efficient methods (next lesson).